- What is Small File Problem
- How it can be fixed using  Databricks OPTIMIZE Command.

- **Optimize** : Delta provides built-in operation OPTIMIZE, it compacts small files into larger more appropriately sized one.
- It uses a **bin_packing** algorithm
- It collects all your file sizes and then sorts it from High to Low.
- Then it starts picking each file and places each of these in a particular bin. Bin is one large file. As long as it fits in the bin.
- Default bin size that delta targets is 1gb. This has proven to be robust.
- In case you still want to change it : **spark.databricks.delta.optimize.maxFileSize** = 

In [0]:
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled",False)
# Check if Adaptive Query Execution Partition Coalescing is active
print(spark.conf.get("spark.sql.adaptive.coalescePartitions.enabled"))

In [0]:
path =  "abfss://labdata@dbdeltalakestorage.dfs.core.windows.net/invoices/invoices_201_99457.parquet"

In [0]:
df = spark.read.parquet(path)
display(df.limit(5))

In [0]:
print(df.count())
print(df.select("category").distinct().count()) #i want to partition by this column

In [0]:
df.groupBy("category").count().display()

In [0]:
df.repartition(5).write.mode("overwrite").partitionBy("category").saveAsTable("deltacatalog.deltadb.optimize_ex1")
#repartition 5 is to mimic small file behaviour

In [0]:
%%time
df_ex1 = spark.read.table("deltacatalog.deltadb.optimize_ex1")
df_out = df_ex1.where("category='Clothing'").collect()

In [0]:
from delta.tables import DeltaTable

table = DeltaTable.forName(spark, "deltacatalog.deltadb.optimize_ex1")
table.optimize().executeCompaction()

In [0]:
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", False)
table.vacuum(0)

In [0]:
%%time
df_ex1 = spark.read.table("deltacatalog.deltadb.optimize_ex1")
df_out = df_ex1.where("category='Clothing'").collect()

In [0]:
from pyspark.sql.functions import *

In [0]:
# run optimize on suppose incremental data coming everyday, run Optimize on particular one date/data
df_fruits = df.filter("category=='Clothing'").withColumn("category",lit("Fruits"))

In [0]:
df_fruits.select("category").distinct().display()

In [0]:
df_fruits.repartition(5).write.mode("overwrite").partitionBy("category").saveAsTable("deltacatalog.deltadb.optimize_ex1")

In [0]:
table.optimize().where("category = 'Fruits'").executeCompaction()

In [0]:
%sql
VACUUM deltacatalog.deltadb.optimize_ex1 RETAIN 0 HOURS;
